In [1]:
import numpy as np
import pandas as pd

# Constants
from scipy.constants import Boltzmann as kB
from scipy.constants import Avogadro as N_A
from scipy.constants import calorie as ca

# Load lammps output

In [2]:
file = f'../../../lammps/CA/fep/convergence.txt'
conv = pd.read_table(file, delimiter=' ', comment='#', header=None, names=['lambda', 'U', 'dU', 'dA', 'V']) 

# FEP calculation

In [3]:
react_c = []
free_en = []
u_lambda = []
kB_cal = kB*N_A / ca / 1e3


for i, (v_lambda, const_lamb_df) in enumerate(conv.groupby('lambda', sort=False)):
    u_lambda.append(const_lamb_df['U'].tolist())
    react_c.append(v_lambda)
    free_en.append(
        -kB_cal*298*np.log(const_lamb_df['dA'][50:].mean()/ const_lamb_df['V'][50:].mean()) 
    )
u_lambda = np.array(u_lambda[:-1])

In [4]:
u_mean = u_lambda[:,50:].mean(axis=1)
DeltaH_01 =  u_mean[0]-u_mean[-1]
free_en = np.array(free_en)
DeltaG_01 = np.nansum(free_en[np.abs(free_en)<np.inf])

## Define the graphene's flat area 

In [5]:
Area = 73*74*1e-20 # m^2

## Compute work of adhesion and water contact angle on graphene 

In [ ]:
Wadh = DeltaG_01/N_A * ca  / (2*Area) * 1e3
gamma_L = 48.1 *1e-3  # for COMPASS water

if 0 <= Wadh/gamma_L <= 2:
    print(f'Contanct angle = {np.rad2deg(np.acos(Wadh/gamma_L -1)):.1f}°')
else:
    print("Arg |arccos| > 1;\nYoung-Dupré formula not appliable.")

Arg |arccos| > 1;
Young-Dupré formula not appliable.
